# Emulators

## Library

In [52]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from UQpy.distributions import Uniform, Normal, JointIndependent #, Lognormal
from UQpy.distributions.collection.Lognormal import Lognormal
from UQpy.surrogates import *

## GLAM testing

In [53]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# ================= FKML: núcleo matemático =================
def _phi(u, lam):
    # Trata lam -> 0 por continuidade: (u^lam - 1)/lam -> ln(u)
    return np.where(np.isclose(lam, 0.0), np.log(u), (u**lam - 1.0)/lam)

def gld_fkml_quantile(u, l1, l2, l3, l4):
    u = np.clip(u, 1e-12, 1-1e-12)
    return l1 + ( _phi(u, l3) - _phi(1-u, l4) ) / l2

def gld_fkml_qprime(u, l2, l3, l4):
    # Q'(u) = (u^(l3-1) + (1-u)^(l4-1)) / l2, com limites log quando lam≈0
    u = np.clip(u, 1e-12, 1-1e-12)
    term1 = np.where(np.isclose(l3, 0.0), 1.0/u, u**(l3 - 1.0))
    term2 = np.where(np.isclose(l4, 0.0), 1.0/(1.0 - u), (1.0 - u)**(l4 - 1.0))
    return (term1 + term2) / l2

# Newton estável para resolver x = Q(u)
def _solve_u_for_x(x, l1, l2, l3, l4, maxit=60, tol=1e-10):
    x = np.atleast_1d(x).astype(float)
    # chute inicial por ranking -> (0,1)
    ranks = (np.argsort(np.argsort(x)) + 0.5) / (len(x) + 1.0)
    u = np.clip(ranks, 1e-4, 1-1e-4)
    for _ in range(maxit):
        q  = gld_fkml_quantile(u, l1, l2, l3, l4)
        qp = gld_fkml_qprime(u, l2, l3, l4)
        step = (q - x)/np.maximum(qp, 1e-16)
        u_new = np.clip(u - step, 1e-8, 1-1e-8)
        if np.max(np.abs(u_new - u)) < tol:
            u = u_new
            break
        u = u_new
    return u

def gld_fkml_pdf(x, l1, l2, l3, l4):
    if l2 <= 0:
        return np.full_like(np.atleast_1d(x), 0.0, dtype=float)
    u  = _solve_u_for_x(x, l1, l2, l3, l4)
    qp = gld_fkml_qprime(u, l2, l3, l4)
    return 1.0/np.maximum(qp, 1e-300)

# ================= MLE com bounds e padronização =================
def _mad(x):
    med = np.median(x)
    return np.median(np.abs(x - med))

def fit_gld_fkml_mle(data, x0=None, bounds=None):
    data = np.asarray(data, dtype=float)
    # Padroniza para estabilizar a busca
    m, s = np.median(data), (1.4826*_mad(data) or np.std(data, ddof=1) or 1.0)
    z = (data - m)/s

    # inicialização
    if x0 is None:
        x0 = np.array([0.0, 1.0, 0.1, 0.1])  # l1≈0, l2≈1, leve assimetria

    # bounds: l2>0; restringe forma para evitar explosões numéricas
    if bounds is None:
        bounds = [(-np.inf, np.inf), (1e-4, np.inf), (-3.0, 3.0), (-3.0, 3.0)]

    def nll(params):
        l1, l2, l3, l4 = params
        if not np.isfinite(l1 + l2 + l3 + l4) or l2 <= 0:
            return np.inf
        f = gld_fkml_pdf(z, l1, l2, l3, l4)
        if np.any(f <= 0) or np.any(~np.isfinite(f)):
            return np.inf
        return -np.sum(np.log(f))

    res = minimize(nll, x0, method="L-BFGS-B", bounds=bounds)

    # desfaz padronização: Qx(u) = m + s * Qz(u)  => l1x = m + s*l1z ; l2x = l2z / s
    l1z, l2z, l3, l4 = res.x
    l1x = m + s*l1z
    l2x = l2z / s
    return (l1x, l2x, l3, l4), res

# ================= Plot APENAS da GLD =================
def plot_gld_only(lambdas, n_points=2000, quantile_trim=1e-3, qp_min=1e-8, ylim=None):
    l1, l2, l3, l4 = lambdas
    a = float(quantile_trim)
    u = np.linspace(a, 1.0 - a, n_points)
    x = gld_fkml_quantile(u, l1, l2, l3, l4)
    qp = gld_fkml_qprime(u, l2, l3, l4)

    mask = np.isfinite(x) & np.isfinite(qp) & (qp > qp_min)
    x, f = x[mask], (1.0/qp[mask])

    order = np.argsort(x)
    x, f = x[order], f[order]

    plt.figure()
    plt.plot(x, f, linewidth=2)
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title("Densidade GLD–FKML")
    if ylim is not None:
        plt.ylim(*ylim)
    plt.tight_layout()
    plt.show()

## Convert lognormal testing

In [3]:
# z1_mu = 2.
# z1_sc = 0.60
# s = np.sqrt(np.log(1 + (z1_sc/z1_mu)**2))
# scale = z1_mu / np.sqrt(1 + (z1_sc/z1_mu)**2)
# print(s, scale)

## Build metamodel to emulator

In [54]:
# r   = Lognormal(s=0.16, loc = 0., scale=4.94)
# s   = Lognormal(s=0.30, loc = 0., scale=1.91)
r   = Normal(loc = 5., scale=0.8)
s   = Normal(loc = 2., scale=0.6)

marg = [r, s]

joint = JointIndependent(marginals=marg)

n_samples = 2000
xx = joint.rvs(n_samples)

y = []
for i in range(n_samples):
    n = 2500
    df = {'r': [xx[i, 0]] * n, 's': [xx[i, 1]] * n}
    df = pd.DataFrame(df)
    z1aux = []
    z2aux = []
    for i in df.iterrows():
        z1_mu = 1.
        z1_sc = 0.028
        s = np.sqrt(np.log(1 + (z1_sc/z1_mu)**2))
        scale = z1_mu / np.sqrt(1 + (z1_sc/z1_mu)**2)
        z1aux.append(stats.lognorm.rvs(s=s, scale=scale, size=1)[0])
        z2_mu = 1.
        z2_sc = 0.096
        s = np.sqrt(np.log(1 + (z2_sc/z2_mu)**2))
        scale = z2_mu / np.sqrt(1 + (z2_sc/z2_mu)**2)
        z2aux.append(stats.lognorm.rvs(s=s, scale=scale, size=1)[0])
    df['z1'] = z1aux
    df['z2'] = z2aux
    df['g'] = df['r'] / df['z1'] - df['s'] * df['z2']
    lambdas, _ = fit_gld_fkml_mle(df['g'].values)
    y.append(lambdas)
    yy = np.array(y)

## x input

In [55]:
xx

array([[5.07365891, 1.35506564],
       [5.1882062 , 1.44627003],
       [4.03060148, 2.20203958],
       ...,
       [5.10681978, 2.81759442],
       [5.82576925, 1.7375507 ],
       [5.29139627, 1.52784602]])

## y input

In [56]:
yy

array([[3.72198114, 7.6230427 , 0.13745197, 0.13777719],
       [3.7510867 , 7.394419  , 0.12003503, 0.11777977],
       [1.84375856, 6.47554121, 0.06078656, 0.14637034],
       ...,
       [2.31541021, 4.90461717, 0.07260106, 0.17695765],
       [4.09073166, 5.99723907, 0.13971783, 0.15406919],
       [3.78199599, 6.96474621, 0.12566785, 0.16727579]])

## Verify xx vars...UQpy generation

In [5]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Criar figura com dois subplots
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# # Histograma da primeira coluna
# ax1.hist(xx[:, 0], bins=15, alpha=0.7, color='blue', edgecolor='black')
# ax1.set_title('Histograma - Coluna 1')
# ax1.set_xlabel('Valores')
# ax1.set_ylabel('Frequência')

# # Histograma da segunda coluna
# ax2.hist(xx[:, 1], bins=15, alpha=0.7, color='red', edgecolor='black')
# ax2.set_title('Histograma - Coluna 2')
# ax2.set_xlabel('Valores')
# ax2.set_ylabel('Frequência')

# plt.tight_layout()
# plt.show()

## Check dimensions

In [57]:
print("xx shape:", xx.shape)
print("yy shape:", yy.shape)

xx shape: (2000, 2)
yy shape: (2000, 4)


## Training

In [58]:
max_degree = 4
polynomial_basis = TotalDegreeBasis(joint, max_degree)

least_squares = LeastSquareRegression()
pce_metamodel = PolynomialChaosExpansion(polynomial_basis=polynomial_basis, regression_method=least_squares)
pce_metamodel.fit(xx, yy)

## Validation

In [100]:
r   = Normal(loc = 5., scale=0.8)
s   = Normal(loc = 2., scale=0.6)

marg = [r, s]

joint = JointIndependent(marginals=marg)

n_samples = 20
xx_val = joint.rvs(n_samples)

y = []
for i in range(n_samples):
    n = 2500
    df = {'r': [xx_val[i, 0]] * n, 's': [xx_val[i, 1]] * n}
    df = pd.DataFrame(df)
    z1aux = []
    z2aux = []
    for i in df.iterrows():
        z1_mu = 1.
        z1_sc = 0.028
        s = np.sqrt(np.log(1 + (z1_sc/z1_mu)**2))
        scale = z1_mu / np.sqrt(1 + (z1_sc/z1_mu)**2)
        z1aux.append(stats.lognorm.rvs(s=s, scale=scale, size=1)[0])
        z2_mu = 1.
        z2_sc = 0.096
        s = np.sqrt(np.log(1 + (z2_sc/z2_mu)**2))
        scale = z2_mu / np.sqrt(1 + (z2_sc/z2_mu)**2)
        z2aux.append(stats.lognorm.rvs(s=s, scale=scale, size=1)[0])
    df['z1'] = z1aux
    df['z2'] = z2aux
    df['g'] = df['r'] / df['z1'] - df['s'] * df['z2']
    lambdas, _ = fit_gld_fkml_mle(df['g'].values)
    y.append(lambdas)
    yy_val = np.array(y)

In [101]:
xx_val

array([[5.80987303, 2.63722869],
       [4.50299466, 2.14873466],
       [5.04601255, 1.57865891],
       [4.05939705, 1.75147992],
       [5.33555466, 2.26667289],
       [3.63617482, 1.24605409],
       [5.13446874, 2.22419067],
       [6.56282229, 1.33393709],
       [3.97205437, 1.45477568],
       [4.83464637, 2.10622044],
       [4.39839561, 1.6451646 ],
       [4.80998371, 2.83750651],
       [4.12659703, 2.02551069],
       [5.36239907, 1.49801584],
       [4.50778025, 1.70334916],
       [6.25474651, 2.16353666],
       [5.55653308, 2.34066071],
       [4.44372481, 2.00418285],
       [5.78069867, 2.64857029],
       [3.93933592, 1.24768713]])

In [102]:
yy_val

array([[3.18352032, 4.90507623, 0.12660776, 0.15677742],
       [2.36095542, 6.01810815, 0.09757127, 0.1686676 ],
       [3.47743686, 6.90282053, 0.10076879, 0.16737362],
       [2.31754097, 7.45921638, 0.11555903, 0.15306609],
       [3.08237429, 5.65656154, 0.10353668, 0.17433741],
       [2.39811922, 9.6544429 , 0.09307679, 0.12388769],
       [2.93065539, 5.90390845, 0.08889519, 0.14702481],
       [5.23841878, 6.44779542, 0.1492522 , 0.16299111],
       [2.5220739 , 7.89995253, 0.12786642, 0.18625704],
       [2.75158328, 6.24621393, 0.09080475, 0.15573432],
       [2.76142995, 7.4891935 , 0.11819012, 0.15170129],
       [1.98310698, 4.90049054, 0.07333024, 0.15702305],
       [2.10783936, 6.84117289, 0.07860114, 0.14056906],
       [3.86869753, 7.25769662, 0.12084842, 0.13135441],
       [2.81808557, 7.15837475, 0.12843281, 0.16083626],
       [4.09737621, 5.46230985, 0.10169868, 0.15243406],
       [3.23194411, 5.00654422, 0.16406042, 0.19242694],
       [2.45547972, 6.55032979,

In [104]:
y_val_pce = pce_metamodel.predict(xx_val)
print("y_val_pce: \n", y_val_pce)

y_val_pce: 
 [[3.18734615 4.91393052 0.09960107 0.16022276]
 [2.36605835 6.08030057 0.10124673 0.16503819]
 [3.47466042 7.11413454 0.11643137 0.14853762]
 [2.31776087 7.28462411 0.10358313 0.16097806]
 [3.08085127 5.58161312 0.10379599 0.15896338]
 [2.39702858 9.43380138 0.10927688 0.15020337]
 [2.92204633 5.71973041 0.10367483 0.16017883]
 [5.23433812 6.62156822 0.12800277 0.13264914]
 [2.52534512 8.30152736 0.10805593 0.15391171]
 [2.73956895 6.04803994 0.1040889  0.16106203]
 [2.76185596 7.37764503 0.10885749 0.15537639]
 [1.9890618  4.86129412 0.09450524 0.17119417]
 [2.11252969 6.50489665 0.10026091 0.16631061]
 [3.8709384  7.08405186 0.12071385 0.1444985 ]
 [2.81325601 7.14732943 0.10881701 0.15581585]
 [4.10186061 5.42850201 0.10887966 0.15016883]
 [3.22831407 5.39496641 0.10327092 0.15811908]
 [2.45043993 6.42509793 0.10300389 0.16291205]
 [3.14694407 4.9066356  0.09938705 0.16058078]
 [2.69833456 9.08181245 0.11258641 0.14794813]]


## Plot